<a href="https://colab.research.google.com/github/arturgabriels779-alt/SISTEMA-PROFISSIONAL---MERCADO-CENTRAL-BH-v4.3/blob/main/mercado_sistema_entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# SISTEMA PROFISSIONAL - MERCADO CENTRAL BH v9.0 final
!pip install streamlit plotly pyngrok pandas openpyxl -q

import os
import time
import threading
from datetime import datetime
import hashlib
import io

# 1. LIMPEZA (Removemos a deleção forçada para preservar dados, usando migração automática)
# for arquivo in ["encomendas.db"]: ... (Comentado para segurança dos dados)

# 2. CÓDIGO DO APP
codigo_app = '''
import streamlit as st
import sqlite3
import pandas as pd
from datetime import datetime
import hashlib
import io

# ========== CONFIGURAÇÃO ==========
st.set_page_config(page_title="Mercado Central BH", page_icon="🍎", layout="wide", initial_sidebar_state="collapsed")

# ========== CSS ADAPTATIVO (DARK MODE NATIVO) ==========
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    * { font-family: 'Inter', -apple-system, sans-serif; }

    /* --- MODO CLARO (Padrão) --- */
    .ios-header {
        background: rgba(255, 255, 255, 0.9);
        backdrop-filter: blur(20px); padding: 25px; border-radius: 24px; margin-bottom: 25px;
        box-shadow: 0 4px 30px rgba(0,0,0,0.05); text-align: center; border: 1px solid #eee;
    }
    .ios-title { font-size: 2.4rem; font-weight: 800; background: linear-gradient(135deg, #007AFF 0%, #5856D6 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin: 0; }

    .ios-card {
        background-color: #FFFFFF;
        border-radius: 20px; padding: 24px; margin-bottom: 20px;
        box-shadow: 0 2px 20px rgba(0,0,0,0.04); border: 1px solid #F5F5F7;
        color: #1C1C1E;
    }
    .ios-card div, .ios-card span, .ios-card p, .ios-card h3 { color: #1C1C1E; }

    .ios-section-title { font-size: 1.4rem; font-weight: 700; color: #1C1C1E; margin: 30px 0 20px 0; padding-left: 15px; border-left: 5px solid #007AFF; }

    /* Inputs no modo claro */
    .stTextInput>div>div>input, .stSelectbox>div>div>select { background-color: #FFFFFF; color: #1C1C1E; border: 1px solid #E5E5EA; }

    /* --- MODO ESCURO (Automático via Media Query) --- */
    @media (prefers-color-scheme: dark) {
        .ios-header {
            background: rgba(30, 30, 30, 0.9);
            border: 1px solid #333;
        }
        .ios-card {
            background-color: #1E1E1E;
            border: 1px solid #333;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            color: #E0E0E0;
        }
        .ios-card div, .ios-card span, .ios-card p, .ios-card h3 { color: #E0E0E0; }

        .ios-section-title { color: #FFFFFF; border-left-color: #0A84FF; }
        .ios-card .sub-text { color: #A0A0A0; }

        .stTextInput>div>div>input, .stSelectbox>div>div>select {
            background-color: #2C2C2E; color: #FFFFFF; border: 1px solid #3A3A3C;
        }
        .stTextInput>div>div>input::placeholder { color: #636366; }
    }

    /* Estilos Gerais */
    .ios-badge { display: inline-block; padding: 6px 14px; border-radius: 20px; font-size: 0.8rem; font-weight: 700; }
    .badge-blue { background: #E5F1FF; color: #007AFF; }
    .badge-green { background: #E8F5E9; color: #2E7D32; }
    @media (prefers-color-scheme: dark) {
        .badge-blue { background: #1C3A5E; color: #5AC8FA; }
        .badge-green { background: #1E3A2C; color: #30D158; }
    }

    .stButton>button { border-radius: 14px; font-weight: 600; height: 48px; }
    .login-container { max-width: 400px; margin: 0 auto; padding-top: 10%; }
</style>
""", unsafe_allow_html=True)

# ========== FUNÇÕES DE SEGURANÇA ==========
def make_hashes(password): return hashlib.sha256(str.encode(password)).hexdigest()
def check_hashes(password, hashed_text): return make_hashes(password) == hashed_text

# ========== BANCO DE DADOS ==========
def get_db():
    conn = sqlite3.connect("encomendas.db", isolation_level=None)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db()
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS clientes (id INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT NOT NULL, telefone TEXT, loja TEXT NOT NULL, corredor TEXT NOT NULL, email TEXT, completo INTEGER DEFAULT 0, ativo INTEGER DEFAULT 1)")
    # Tabela encomendas SEM a coluna nova inicialmente para permitir migração
    c.execute("CREATE TABLE IF NOT EXISTS encomendas (id TEXT PRIMARY KEY, numero_rastreio TEXT UNIQUE NOT NULL, data_chegada TIMESTAMP DEFAULT CURRENT_TIMESTAMP, logista_id INTEGER, nome_cliente_manual TEXT, loja TEXT NOT NULL, corredor TEXT NOT NULL, volumes INTEGER DEFAULT 1, status TEXT DEFAULT 'CENTRAL DE SEGURANÇA', data_retirada TIMESTAMP, quem_retirou TEXT, documento TEXT, justificativa_devolucao TEXT, responsavel TEXT NOT NULL, observacoes TEXT, transportadora TEXT, peso_aproximado TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS usuarios (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, password TEXT, real_name TEXT, role TEXT DEFAULT 'user', cargo TEXT, telefone TEXT, ativo INTEGER DEFAULT 1)")

    # === MIGRAÇÃO AUTOMÁTICA (ADICIONA COLUNA SE NÃO EXISTIR) ===
    try:
        c.execute("SELECT responsavel_entrega FROM encomendas LIMIT 1")
    except sqlite3.OperationalError:
        c.execute("ALTER TABLE encomendas ADD COLUMN responsavel_entrega TEXT")

    if c.execute("SELECT COUNT(*) FROM clientes").fetchone()[0] == 0:
        c.execute("INSERT INTO clientes (nome, telefone, loja, corredor, email, completo) VALUES ('Mercado Exemplo', '31999999999', 'Loja 01', 'A', 'teste@teste.com', 1)")

    if c.execute("SELECT COUNT(*) FROM usuarios").fetchone()[0] == 0:
        admin_pass = make_hashes("admin123")
        func_pass = make_hashes("func123")
        c.execute("INSERT INTO usuarios (username, password, real_name, role, cargo, telefone, ativo) VALUES (?, ?, ?, 'admin', 'Gerente Sistema', '31900000000', 1)", ("admin", admin_pass, "Administrador"))
        c.execute("INSERT INTO usuarios (username, password, real_name, role, cargo, telefone, ativo) VALUES (?, ?, ?, 'user', 'Auxiliar', '31911111111', 1)", ("funcio", func_pass, "João Funcio"))

    conn.commit()
    conn.close()

if 'db_ok' not in st.session_state:
    init_db()
    st.session_state['db_ok'] = True

# ========== CONTROLE DE SESSÃO ==========
if 'authenticated' not in st.session_state:
    st.session_state['authenticated'] = False
    st.session_state['username'] = ""
    st.session_state['real_name'] = ""
    st.session_state['role'] = ""
    st.session_state['user_id'] = None

# ========== TELA DE LOGIN ==========
def login_screen():
    st.markdown('<div class="login-container"><div style="text-align:center; font-size:4rem;">🍎</div></div>', unsafe_allow_html=True)
    st.markdown('<div class="ios-card">', unsafe_allow_html=True)
    st.markdown("<h2 style='text-align: center; margin-bottom: 20px;'>Mercado Central BH</h2>", unsafe_allow_html=True)

    with st.form("login_form"):
        username = st.text_input("Usuário", placeholder="Digite seu usuário")
        password = st.text_input("Senha", type="password", placeholder="Digite sua senha")

        if st.form_submit_button("Entrar", use_container_width=True, type="primary"):
            conn = get_db()
            user = conn.execute("SELECT * FROM usuarios WHERE username = ?", (username,)).fetchone()
            conn.close()

            if user and check_hashes(password, user['password']):
                if user['ativo'] == 0:
                    st.error("❌ Este usuário está inativo.")
                else:
                    st.session_state['authenticated'] = True
                    st.session_state['username'] = username
                    st.session_state['real_name'] = user['real_name']
                    st.session_state['role'] = user['role']
                    st.session_state['user_id'] = user['id']
                    st.rerun()
            else:
                st.error("❌ Usuário ou senha incorretos.")

    st.markdown('</div>', unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; color: gray; font-size: 0.8rem; margin-top: 20px;'>Admin: admin/admin123<br>Func: funcio/func123</p>", unsafe_allow_html=True)

# ========== TELA PRINCIPAL ==========
if not st.session_state['authenticated']:
    login_screen()
else:
    # HEADER
    col_t, col_u, col_p, col_l = st.columns([4, 1, 1, 1])
    with col_t:
        st.markdown('<div class="ios-header" style="margin-bottom:0"><h1 class="ios-title">🍎 Mercado Central BH</h1></div>', unsafe_allow_html=True)
    with col_u:
        st.markdown(f"<div style='padding-top: 30px; text-align: right; font-weight: 500;'>👤 {st.session_state['real_name']}</div>", unsafe_allow_html=True)
    with col_p:
        if st.button("⚙️ Conta", key="btn_profile"):
            st.session_state['page'] = 'perfil'
            st.rerun()
    with col_l:
        if st.button("🚪 Sair", key="logout_btn"):
            st.session_state['authenticated'] = False
            st.rerun()

    # MENU
    menu_base = {"🏠 Início": "home", "📦 Nova": "nova", "✅ Status": "status", "👥 Clientes": "clientes", "👷 Equipe": "equipe", "📊 Relatórios": "relatorios"}
    if st.session_state['role'] == 'admin':
        menu_base["🔒 Admin"] = "admin_panel"

    cols = st.columns(len(menu_base))
    for i, (nome, chave) in enumerate(menu_base.items()):
        if cols[i].button(nome, use_container_width=True, key=f"nav_{chave}"):
            st.session_state['page'] = chave
            st.rerun()

    if 'page' not in st.session_state: st.session_state['page'] = 'home'
    page = st.session_state['page']

    # ========== FUNÇÕES AUXILIARES ==========
    def badge(status):
        cores = {'CENTRAL DE SEGURANÇA': 'badge-blue', 'ENTREGUE': 'badge-green', 'AGUARDANDO DEVOLUÇÃO': 'badge-orange'}
        return f"<span class='ios-badge {cores.get(status, '')}'>{status}</span>"

    # ========== PÁGINA: MINHA CONTA ==========
    if page == "perfil":
        st.markdown('<div class="ios-section-title">⚙️ Minha Conta</div>', unsafe_allow_html=True)
        user_id = st.session_state['user_id']
        conn = get_db(); user_data = conn.execute("SELECT * FROM usuarios WHERE id=?", (user_id,)).fetchone(); conn.close()

        st.markdown('<div class="ios-card">', unsafe_allow_html=True)
        st.subheader("Editar Dados")
        with st.form("edit_profile"):
            st.text_input("Nome", value=user_data['real_name'], disabled=True)
            st.text_input("Usuário", value=user_data['username'], disabled=True)
            n_tel = st.text_input("Telefone de Contato", value=user_data['telefone'] or "")
            st.markdown("---")
            st.markdown("**Alterar Senha** (deixe em branco para manter a atual)")
            n_pass = st.text_input("Nova Senha", type="password")
            n_pass_conf = st.text_input("Confirmar Nova Senha", type="password")

            if st.form_submit_button("Salvar Alterações", type="primary"):
                conn = get_db(); c = conn.cursor()
                c.execute("UPDATE usuarios SET telefone=? WHERE id=?", (n_tel, user_id))
                if n_pass:
                    if n_pass == n_pass_conf:
                        h_pass = make_hashes(n_pass)
                        c.execute("UPDATE usuarios SET password=? WHERE id=?", (h_pass, user_id))
                    else:
                        st.error("❌ As senhas não coincidem."); conn.close(); st.stop()
                conn.commit(); conn.close()
                st.success("✅ Dados atualizados!")
        st.markdown('</div>', unsafe_allow_html=True)

    # ========== HOME ==========
    elif page == "home":
        st.markdown('<div class="ios-section-title">📊 Visão Geral</div>', unsafe_allow_html=True)
        conn = get_db()
        central = conn.execute("SELECT COUNT(*) FROM encomendas WHERE status='CENTRAL DE SEGURANÇA'").fetchone()[0]
        entregues = conn.execute("SELECT COUNT(*) FROM encomendas WHERE status='ENTREGUE'").fetchone()[0]
        conn.close()

        c1, c2 = st.columns(2)
        c1.markdown(f"<div class='ios-card' style='text-align:center; padding:30px;'><div style='font-size:2.5rem; font-weight:800; color:#007AFF'>{central}</div><div class='sub-text'>🛡️ Na Central</div></div>", unsafe_allow_html=True)
        c2.markdown(f"<div class='ios-card' style='text-align:center; padding:30px;'><div style='font-size:2.5rem; font-weight:800; color:#34C759'>{entregues}</div><div class='sub-text'>✅ Entregues</div></div>", unsafe_allow_html=True)

        st.markdown('<div class="ios-section-title">🕓 Últimas Encomendas</div>', unsafe_allow_html=True)
        conn = get_db()
        recentes = conn.execute("SELECT e.*, c.nome as cliente_nome FROM encomendas e LEFT JOIN clientes c ON e.logista_id = c.id ORDER BY e.rowid DESC LIMIT 10").fetchall()
        conn.close()

        for r in recentes:
            nome_exibicao = r['cliente_nome'] or r['nome_cliente_manual'] or "N/A"
            col_info, col_btn = st.columns([5, 1])
            with col_info:
                st.markdown(f"""
                    <div class='ios-card' style='margin-bottom:0px; padding:15px;'>
                        <div style="font-weight:700;">📦 {r['numero_rastreio']} <span style="font-weight:400; font-size:0.9rem; opacity:0.7;">({nome_exibicao})</span></div>
                        <div class='sub-text' style="font-size:0.85rem;">📍 {r['loja']} • {r['corredor']} | {badge(r['status'])}</div>
                    </div>
                """, unsafe_allow_html=True)
            with col_btn:
                if st.button("➡️", key=f"go_{r['id']}"):
                    st.session_state['buscar_id'] = r['numero_rastreio']
                    st.session_state['page'] = 'status'
                    st.rerun()
            st.write("")

    # ========== NOVA ENCOMENDA ==========
    elif page == "nova":
        st.markdown('<div class="ios-section-title">📦 Nova Encomenda</div>', unsafe_allow_html=True)
        st.info(f"🔐 Registrado por: **{st.session_state['real_name']}**")

        st.markdown('<div class="ios-card">', unsafe_allow_html=True)
        tipo = st.radio("Tipo de Destinatário", ["Cliente Cadastrado", "Cliente Avulso"], horizontal=True)

        conn = get_db()
        clientes_db = conn.execute("SELECT id, nome, loja, corredor FROM clientes WHERE ativo=1 ORDER BY nome").fetchall()
        conn.close()

        cliente_sel_id = None
        loja_auto = ""
        corredor_auto = ""

        if tipo == "Cliente Cadastrado":
            opcoes = {f"{c['nome']} ({c['loja']})": c for c in clientes_db}
            sel = st.selectbox("Selecione o Cliente", [""] + list(opcoes.keys()))
            if sel:
                cliente = opcoes[sel]
                cliente_sel_id = cliente['id']
                loja_auto = cliente['loja']
                corredor_auto = cliente['corredor']
        st.markdown('</div>', unsafe_allow_html=True)

        with st.form("form_nova"):
            col1, col2 = st.columns(2)
            rastreio = col1.text_input("Nº Rastreio *")
            transp = col2.selectbox("Transportadora *", ["Correios", "Jadlog", "Loggi", "Total Express", "Outra"])

            nome_avulso = ""
            tel_avulso = ""
            if tipo == "Cliente Avulso":
                c_nome, c_tel = st.columns([3, 1])
                nome_avulso = c_nome.text_input("Nome do Cliente *")
                tel_avulso = c_tel.text_input("Telefone")

            col_l, col_c = st.columns(2)
            loja = col_l.text_input("Loja/Box *", value=loja_auto)
            corredor = col_c.text_input("Corredor *", value=corredor_auto)

            col_v, col_p = st.columns(2)
            volumes = col_v.number_input("Volumes", min_value=1, value=1)
            peso = col_p.selectbox("Peso", ["Não informado", "Até 5kg", "5-15kg", "+15kg"])

            obs = st.text_area("Observações")

            if st.form_submit_button("💾 Salvar Encomenda", type="primary", use_container_width=True):
                if not rastreio or not loja or not corredor or (tipo == "Cliente Avulso" and not nome_avulso):
                    st.error("❌ Preencha todos os campos obrigatórios (*).")
                else:
                    try:
                        conn = get_db()
                        c = conn.cursor()
                        final_cliente_id = cliente_sel_id

                        if tipo == "Cliente Avulso":
                            existente = c.execute("SELECT id FROM clientes WHERE nome = ? AND loja = ?", (nome_avulso, loja)).fetchone()
                            if existente:
                                final_cliente_id = existente[0]
                            else:
                                c.execute("INSERT INTO clientes (nome, telefone, loja, corredor, email, completo, ativo) VALUES (?,?,?,?,?,?,1)", (nome_avulso, tel_avulso, loja, corredor, "", 0))
                                final_cliente_id = c.lastrowid

                        novo_id = f"MCBH-{datetime.now().strftime('%Y%m%d%H%M%S')}"
                        c.execute("INSERT INTO encomendas (id, numero_rastreio, logista_id, loja, corredor, volumes, responsavel, transportadora, peso_aproximado, observacoes) VALUES (?,?,?,?,?,?,?,?,?,?)",
                                 (novo_id, rastreio, final_cliente_id, loja, corredor, volumes, st.session_state['real_name'], transp, peso, obs))

                        conn.commit()
                        conn.close()
                        st.success(f"✅ Encomenda registrada!")
                        st.balloons()

                    except sqlite3.IntegrityError:
                        st.error("❌ Erro: Este número de rastreio já existe.")

    # ========== STATUS ==========
    elif page == "status":
        st.markdown('<div class="ios-section-title">✅ Atualizar Status</div>', unsafe_allow_html=True)

        default_val = st.session_state.pop('buscar_id', "")
        busca = st.text_input("🔍 Buscar Rastreio ou ID", value=default_val)

        if busca:
            conn = get_db()
            res = conn.execute("SELECT e.*, c.nome as cliente_nome FROM encomendas e LEFT JOIN clientes c ON e.logista_id = c.id WHERE e.numero_rastreio=? OR e.id=?", (busca,busca)).fetchone()
            conn.close()
            if res:
                st.session_state['enc_atual'] = dict(res)

        if 'enc_atual' in st.session_state and st.session_state['enc_atual']:
            enc = st.session_state['enc_atual']
            nome_cli = enc['cliente_nome'] or enc['nome_cliente_manual'] or "N/A"

            st.markdown(f"""
                <div class='ios-card'>
                    <h3>{enc['numero_rastreio']}</h3>
                    <p>👤 {nome_cli} | 📍 {enc['loja']} - {enc['corredor']}</p>
                    <p style="font-size:0.9rem; opacity:0.8;">Recebido por: <b>{enc['responsavel']}</b></p>
                </div>
            """, unsafe_allow_html=True)

            novo_status = st.selectbox("Novo Status", ["CENTRAL DE SEGURANÇA", "ENTREGUE", "AGUARDANDO DEVOLUÇÃO", "DEVOLVIDO"])

            with st.form("form_status"):
                quem = None; doc = None; motivo = None
                if novo_status == "ENTREGUE":
                    quem = st.text_input("Quem Retirou *")
                    doc = st.text_input("Documento *")
                elif novo_status == "DEVOLVIDO":
                    motivo = st.selectbox("Motivo", ["Cliente não encontrado", "Endereço errado", "Recusado"])

                obs = st.text_area("Observações", value=enc['observacoes'] or "")

                if st.form_submit_button("💾 Atualizar", type="primary"):
                    conn = get_db()
                    # LÓGICA ALTERADA: Salva 'responsavel_entrega' se for ENTREGUE, limpa nos outros casos
                    if novo_status == "ENTREGUE" and quem and doc:
                        user_entregou = st.session_state['real_name']
                        conn.execute("UPDATE encomendas SET status=?, data_retirada=datetime('now'), quem_retirou=?, documento=?, observacoes=?, responsavel_entrega=? WHERE id=?", (novo_status, quem, doc, obs, user_entregou, enc['id']))
                    elif novo_status == "DEVOLVIDO":
                        conn.execute("UPDATE encomendas SET status=?, justificativa_devolucao=?, observacoes=?, responsavel_entrega=? WHERE id=?", (novo_status, motivo, obs, None, enc['id']))
                    else:
                        conn.execute("UPDATE encomendas SET status=?, observacoes=?, responsavel_entrega=? WHERE id=?", (novo_status, obs, None, enc['id']))
                    conn.commit()
                    conn.close()
                    st.success("✅ Atualizado!")
                    del st.session_state['enc_atual']
                    st.rerun()

    # ========== CLIENTES ==========
    elif page == "clientes":
        st.markdown('<div class="ios-section-title">👥 Clientes</div>', unsafe_allow_html=True)
        editing_id = st.session_state.get('edit_cliente_id', None)

        if editing_id:
            conn = get_db(); cliente_atual = conn.execute("SELECT * FROM clientes WHERE id = ?", (editing_id,)).fetchone(); conn.close()
            if cliente_atual:
                st.markdown('<div class="ios-card">', unsafe_allow_html=True); st.subheader(f"✏️ Editando: {cliente_atual['nome']}")
                with st.form("form_edit_cliente"):
                    n_nome = st.text_input("Nome", value=cliente_atual['nome']); n_tel = st.text_input("Telefone", value=cliente_atual['telefone'])
                    n_loja = st.text_input("Loja", value=cliente_atual['loja']); n_corr = st.text_input("Corredor", value=cliente_atual['corredor'])
                    atualizar_tudo = st.checkbox("🔄 Aplicar alterações de endereço a TODAS as encomendas", value=False)
                    col_b1, col_b2 = st.columns(2)
                    with col_b1:
                        if st.form_submit_button("💾 Salvar", type="primary"):
                            conn = get_db(); c = conn.cursor()
                            c.execute("UPDATE clientes SET nome=?, telefone=?, loja=?, corredor=? WHERE id=?", (n_nome, n_tel, n_loja, n_corr, editing_id))
                            if atualizar_tudo: c.execute("UPDATE encomendas SET loja=?, corredor=? WHERE logista_id=?", (n_loja, n_corr, editing_id))
                            conn.commit(); conn.close(); st.session_state['edit_cliente_id'] = None; st.rerun()
                    with col_b2:
                        if st.form_submit_button("❌ Cancelar"): st.session_state['edit_cliente_id'] = None; st.rerun()
                st.markdown('</div>', unsafe_allow_html=True)
        else:
            conn = get_db(); clientes = conn.execute("SELECT * FROM clientes WHERE ativo=1 ORDER BY nome").fetchall(); conn.close()
            for c in clientes:
                col_info, col_edit = st.columns([5, 1])
                with col_info: st.markdown(f"<div class='ios-card' style='margin-bottom:0px; padding:15px;'><b>{c['nome']}</b><br><span style='opacity:0.7'>{c['loja']} | {c['telefone']}</span></div>", unsafe_allow_html=True)
                with col_edit:
                    if st.button("✏️", key=f"edc_{c['id']}"): st.session_state['edit_cliente_id'] = c['id']; st.rerun()
                st.write("")

    # ========== EQUIPE ==========
    elif page == "equipe":
        st.markdown('<div class="ios-section-title">👷 Equipe & Usuários</div>', unsafe_allow_html=True)
        is_admin = (st.session_state['role'] == 'admin')
        editing_user = st.session_state.get('edit_func_id', None)

        if editing_user and is_admin:
            conn = get_db(); user = conn.execute("SELECT * FROM usuarios WHERE id=?", (editing_user,)).fetchone(); conn.close()
            if user:
                st.markdown('<div class="ios-card">', unsafe_allow_html=True); st.subheader(f"✏️ Editando: {user['real_name']}")
                with st.form("form_edit_user"):
                    n_nome = st.text_input("Nome Completo", value=user['real_name'])
                    n_cargo = st.text_input("Cargo/Função", value=user['cargo'] or "")
                    n_tel = st.text_input("Telefone", value=user['telefone'] or "")
                    n_role = st.selectbox("Nível de Acesso", ["user", "admin"], index=0 if user['role']=="user" else 1)
                    col1, col2 = st.columns(2)
                    with col1:
                        if st.form_submit_button("💾 Salvar", type="primary"):
                            conn = get_db(); conn.execute("UPDATE usuarios SET real_name=?, cargo=?, telefone=?, role=? WHERE id=?", (n_nome, n_cargo, n_tel, n_role, editing_user)); conn.commit(); conn.close(); st.session_state['edit_func_id'] = None; st.rerun()
                    with col2:
                        if st.form_submit_button("❌ Cancelar"): st.session_state['edit_func_id'] = None; st.rerun()
                st.markdown('</div>', unsafe_allow_html=True)
        else:
            conn = get_db(); users = conn.execute("SELECT * FROM usuarios ORDER BY ativo DESC, real_name").fetchall(); conn.close()
            for u in users:
                status_icone = "✅" if u['ativo'] == 1 else "🚫"
                admin_badge = "🔒 Admin" if u['role'] == 'admin' else ""
                col_info, col_actions = st.columns([4, 2])
                with col_info:
                    st.markdown(f"""
                        <div class='ios-card' style='margin-bottom:0px; padding:15px;'>
                            <div style="font-weight:700;">{status_icone} {u['real_name']} <span style="font-size:0.8rem; color:#007AFF;">{admin_badge}</span></div>
                            <div style="opacity:0.7; font-size:0.9rem;">📂 {u['cargo'] or 'N/A'} | 📱 {u['telefone'] or 'N/A'} | 👤 Login: {u['username']}</div>
                        </div>
                    """, unsafe_allow_html=True)
                if is_admin:
                    with col_actions:
                        c_edit, c_del = st.columns(2)
                        if c_edit.button("✏️", key=f"edu_{u['id']}"): st.session_state['edit_func_id'] = u['id']; st.rerun()
                        if u['username'] != 'admin':
                            if u['ativo'] == 1:
                                if c_del.button("🗑️", key=f"delu_{u['id']}"): conn = get_db(); conn.execute("UPDATE usuarios SET ativo=0 WHERE id=?", (u['id'],)); conn.commit(); conn.close(); st.warning("Inativado."); st.rerun()
                            else:
                                if c_del.button("♻️", key=f"actu_{u['id']}"): conn = get_db(); conn.execute("UPDATE usuarios SET ativo=1 WHERE id=?", (u['id'],)); conn.commit(); conn.close(); st.success("Reativado."); st.rerun()
                st.write("")

    # ========== PAINEL ADMIN ==========
    elif page == "admin_panel" and st.session_state['role'] == 'admin':
        st.markdown('<div class="ios-section-title">🔒 Painel Administrativo</div>', unsafe_allow_html=True)
        with st.expander("➕ Criar Novo Usuário/Funcionário", expanded=True):
            with st.form("new_user_form"):
                col1, col2, col3 = st.columns(3)
                u_login = col1.text_input("Login de Acesso *")
                u_nome = col2.text_input("Nome Completo *")
                u_tel = col3.text_input("Telefone")
                col4, col5, col6 = st.columns(3)
                u_pass = col4.text_input("Senha Inicial *", type="password")
                u_cargo = col5.selectbox("Cargo/Função", ["Aprendiz", "Auxiliar", "Colaborador", "Supervisor", "Gerente", "Outro"])
                u_role = col6.selectbox("Nível de Acesso", ["user", "admin"])

                if st.form_submit_button("Criar Usuário", type="primary"):
                    if u_login and u_nome and u_pass:
                        try:
                            conn = get_db(); h_pass = make_hashes(u_pass)
                            conn.execute("INSERT INTO usuarios (username, password, real_name, role, cargo, telefone, ativo) VALUES (?,?,?,?,?,?,1)", (u_login, h_pass, u_nome, u_role, u_cargo, u_tel))
                            conn.commit(); conn.close(); st.success(f"✅ Usuário {u_nome} criado!"); st.rerun()
                        except: st.error("❌ Login já existe.")

    # ========== RELATÓRIOS ==========
    elif page == "relatorios":
        st.markdown('<div class="ios-section-title">📊 Relatórios</div>', unsafe_allow_html=True)

        conn = get_db()
        # Query base (Sem coluna 'Entregue por')
        query = """
            SELECT
                e.id as "ID",
                e.numero_rastreio as "Rastreio",
                c.nome as "Cliente",
                e.loja as "Loja",
                e.corredor as "Corredor",
                e.volumes as "Volumes",
                e.status as "Status",
                e.responsavel as "Recebido Por",
                e.data_chegada as "Data Chegada",
                e.data_retirada as "Data Entrega",
                e.quem_retirou as "Retirado Por",
                e.transportadora as "Transportadora"
            FROM encomendas e
            LEFT JOIN clientes c ON e.logista_id = c.id
        """

        # Query Admin (COM coluna 'Entregue por')
        if st.session_state['role'] == 'admin':
            query = """
                SELECT
                    e.id as "ID",
                    e.numero_rastreio as "Rastreio",
                    c.nome as "Cliente",
                    e.loja as "Loja",
                    e.corredor as "Corredor",
                    e.volumes as "Volumes",
                    e.status as "Status",
                    e.responsavel as "Recebido Por",
                    e.responsavel_entrega as "Entregue por",
                    e.data_chegada as "Data Chegada",
                    e.data_retirada as "Data Entrega",
                    e.quem_retirou as "Retirado Por",
                    e.transportadora as "Transportadora"
                FROM encomendas e
                LEFT JOIN clientes c ON e.logista_id = c.id
            """

        df = pd.read_sql_query(query, conn)
        conn.close()

        st.dataframe(df, use_container_width=True)

        st.markdown("---")
        st.markdown("### 📥 Exportar Dados")
        col_csv, col_xls = st.columns(2)

        with col_csv:
            csv = df.to_csv(sep=';', index=False).encode('utf-8-sig')
            st.download_button(
                label="📄 Baixar CSV (Excel BR)",
                data=csv,
                file_name="relatorio_mercado.csv",
                mime='text/csv'
            )

        with col_xls:
            buffer = io.BytesIO()
            with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
                df.to_excel(writer, index=False, sheet_name='Encomendas')
            buffer.seek(0)

            st.download_button(
                label="📊 Baixar Excel (.xlsx)",
                data=buffer,
                file_name="relatorio_mercado.xlsx",
                mime='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
            )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(codigo_app)

print("✅ Sistema atualizado com verificação automática de banco de dados.")

# 3. EXECUÇÃO
def run():
    os.system("streamlit run app.py --server.port 8501 --server.headless true")

t = threading.Thread(target=run)
t.start()
time.sleep(5)

# 4. NGROK
from pyngrok import ngrok, conf
conf.get_default().auth_token = "38ybdCOek4nliyp86IxOLXqGstw_5RBMbbZCPKeyvkTCwaVgb"

try:
    ngrok.kill()
    url = ngrok.connect(8501, "http")
    print("\n" + "="*60)
    print(f"🚀 SISTEMA ONLINE: {url}")
    print("="*60)
    print("\n💡 CORREÇÃO APLICADA: O banco de dados foi atualizado automaticamente.")
    print("💡 A coluna 'Entregue por' agora mostrará quem fez a entrega (apenas Admin).")
except Exception as e:
    print(f"Erro ao gerar link: {e}")

✅ Sistema atualizado com verificação automática de banco de dados.

🚀 SISTEMA ONLINE: NgrokTunnel: "https://abysmally-judicable-veronica.ngrok-free.dev" -> "http://localhost:8501"

💡 CORREÇÃO APLICADA: O banco de dados foi atualizado automaticamente.
💡 A coluna 'Entregue por' agora mostrará quem fez a entrega (apenas Admin).
